# Part 5 — Guardrail Moderation Pipeline

Self-contained notebook. Re-runs the data load and (where needed) reloads the saved baseline checkpoint so it can execute standalone on Colab.

In [ ]:
!pip install -q transformers torch scikit-learn fairlearn aif360 pandas matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 42
np.random.seed(SEED)

USE_COLS = ["comment_text", "toxic", "black", "white", "muslim", "jewish"]
IDENTITY_COLS = ["black", "white", "muslim", "jewish"]

raw = pd.read_csv("data/jigsaw-unintended-bias-train.csv", usecols=USE_COLS)
raw["label"] = (raw["toxic"] >= 0.5).astype(int)

sample = raw.sample(n=120_000, random_state=SEED)
train_df, eval_df = train_test_split(
    sample, test_size=20_000, stratify=sample["label"], random_state=SEED,
)
train_df = train_df.reset_index(drop=True)
eval_df = eval_df.reset_index(drop=True)
print("train:", train_df.shape, "eval:", eval_df.shape)


In [ ]:
import os, glob
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 128

# Prefer best mitigated checkpoint if present (Part 5 scenario), else baseline,
# else fall back to the pretrained backbone.
_candidates = (sorted(glob.glob("checkpoints/best_mitigated_*"))
               + ["checkpoints/baseline"])
CKPT_DIR = next((c for c in _candidates if os.path.isdir(c)), MODEL_NAME)
print("loading model from", CKPT_DIR)

tokenizer = AutoTokenizer.from_pretrained(CKPT_DIR)
model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR, num_labels=2)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

class ToxicDataset(Dataset):
    def __init__(self, df):
        enc = tokenizer(df["comment_text"].astype(str).tolist(),
                        truncation=True, padding="max_length", max_length=MAX_LEN)
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.labels = df["label"].tolist()
    def __len__(self): return len(self.labels)
    def __getitem__(self, i):
        return {
            "input_ids": torch.tensor(self.input_ids[i]),
            "attention_mask": torch.tensor(self.attention_mask[i]),
            "labels": torch.tensor(self.labels[i]),
        }

train_ds = ToxicDataset(train_df)
eval_ds = ToxicDataset(eval_df)


In [ ]:
# Score the eval set so downstream cells can reuse predictions.
from transformers import Trainer, TrainingArguments

_args = TrainingArguments(output_dir="tmp_eval", per_device_eval_batch_size=64,
                          report_to="none")
_trainer = Trainer(model=model, args=_args)
_logits = _trainer.predict(eval_ds).predictions
eval_probs = torch.softmax(torch.tensor(_logits), dim=-1)[:, 1].numpy()
eval_preds = (eval_probs >= 0.5).astype(int)
eval_df = eval_df.assign(prob=eval_probs, pred=eval_preds)
trainer = _trainer  # downstream cells reference `trainer` and `metrics`
metrics = _trainer.evaluate(eval_ds)
print({k: round(v, 4) for k, v in metrics.items() if k.startswith("eval_")})


In [ ]:
# Shared helpers reused across multiple parts (defined once here so each
# notebook is fully standalone).
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix, precision_score, recall_score)

def compute_metrics(pred):
    logits, labels = pred
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
        "auc_roc": roc_auc_score(labels, probs),
    }

def per_cohort_metrics(df):
    y, p = df["label"].values, df["pred"].values
    tn, fp, fn, tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
    return {
        "n": len(df),
        "TPR": tp / (tp + fn) if (tp + fn) else 0.0,
        "FPR": fp / (fp + tn) if (fp + tn) else 0.0,
        "FNR": fn / (tp + fn) if (tp + fn) else 0.0,
        "Precision": precision_score(y, p, zero_division=0),
    }


In [ ]:
import re
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator  # sklearn >=1.6; fallback below if unavailable

BLOCKLIST = {
    # 5 patterns; one uses a capturing group over a verb set per spec.
    "threats": [
        re.compile(r"\bi\s*(?:'ll|will|am gonna|am going to|gonna)\s+(kill|murder|shoot|stab|hurt)\s+you\b", re.I),
        re.compile(r"\byou(?:'re| are)\s+going to die\b", re.I),
        re.compile(r"\b(?:someone should|i hope someone)\s+(?:kill|shoot|hurt|stab)s?\s+(?:you|him|her|them)\b", re.I),
        re.compile(r"\bi('ll| will)\s+find (?:where you live|out where you are)\b", re.I),
        re.compile(r"\byou (?:deserve to|should) (?:be )?(?:beaten|hurt|killed)\b", re.I),
    ],
    # 4 patterns; second-person directed only, never first-person.
    "self_harm": [
        re.compile(r"\b(?:go\s+)?kill yourself\b", re.I),
        re.compile(r"\byou should (?:kill|hurt|harm) yourself\b", re.I),
        re.compile(r"\bnobody would miss you\b", re.I),
        re.compile(r"\bdo (?:everyone|us all) a favou?r and (?:disappear|die|leave)\b", re.I),
    ],
    # 4 patterns; first-person + private-info knowledge.
    "doxxing": [
        re.compile(r"\bi(?:'ll| will)?\s*(?:post|leak|share|dox)\s+your\s+(?:address|phone|info|details)\b", re.I),
        re.compile(r"\bi know where you (?:live|work)\b", re.I),
        re.compile(r"\bi found your (?:real )?(?:name|address|workplace|family)\b", re.I),
        re.compile(r"\beveryone will know who you (?:really )?are\b", re.I),
    ],
    # 4 patterns; non-capturing variant group per spec.
    "dehumanization": [
        re.compile(r"\b(\w+)\s+are\s+not\s+(?:human|people|person)s?\b", re.I),
        re.compile(r"\b(\w+)\s+are\s+animals\b", re.I),
        re.compile(r"\b(\w+)\s+(?:should be|deserve to be)\s+exterminated\b", re.I),
        re.compile(r"\b(\w+)\s+are\s+a\s+disease\b", re.I),
    ],
    # 3 patterns; one uses a lookahead per spec.
    "harassment": [
        re.compile(r"\b(?:everyone|let'?s all)\s+(?:report|attack|raid|target)\b(?=\s+@?\w+)", re.I),
        re.compile(r"\bmass report (?:this|that) (?:account|user|person)\b", re.I),
        re.compile(r"\b(?:go after|raid)\s+(?:their\s+(?:profile|account)|@?\w+)\b", re.I),
    ],
}
print("blocklist patterns:",
      {cat: len(pats) for cat, pats in BLOCKLIST.items()},
      "total:", sum(len(p) for p in BLOCKLIST.values()))

def regex_filter(text: str):
    """Returns (category, matched_pattern) or (None, None)."""
    for category, patterns in BLOCKLIST.items():
        for pat in patterns:
            if pat.search(text):
                return category, pat.pattern
    return None, None

# Smoke test
for sample in ["I will kill you", "go kill yourself", "i know where you live",
               "they are not human", "everyone report @bob", "this is a normal comment"]:
    print(regex_filter(sample), "<-", sample)

In [ ]:
# Calibrate the fine-tuned DistilBERT with isotonic regression on a held-out
# slice of the eval set, using sklearn's IsotonicRegression directly. This
# avoids the sklearn >=1.6 ClassifierMixin/FrozenEstimator tag plumbing
# required by CalibratedClassifierCV.
from sklearn.isotonic import IsotonicRegression

def distilbert_probs(texts, batch_size=64):
    model.eval()
    device = next(model.parameters()).device
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            enc = tokenizer(list(texts[i:i+batch_size]), truncation=True,
                            padding=True, max_length=MAX_LEN,
                            return_tensors="pt").to(device)
            logits = model(**enc).logits
            out.extend(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.array(out)

calib_df = eval_df.sample(n=2000, random_state=SEED)
raw_probs = distilbert_probs(calib_df["comment_text"].astype(str).tolist())
isotonic = IsotonicRegression(out_of_bounds="clip").fit(raw_probs, calib_df["label"].values)

def calibrated_prob(text: str) -> float:
    raw = distilbert_probs([text])[0]
    return float(isotonic.predict([raw])[0])

print(f"calibrator fitted on {len(calib_df)} samples")
print("sample calibration:", round(calibrated_prob("you are awful"), 3))

In [ ]:
# Persist the isotonic calibrator so pipeline.py can reload it later.
import pickle, os
os.makedirs(CKPT_DIR, exist_ok=True)
with open(os.path.join(CKPT_DIR, "isotonic.pkl"), "wb") as f:
    pickle.dump(isotonic, f)
print("saved", os.path.join(CKPT_DIR, "isotonic.pkl"))

In [ ]:
class ModerationPipeline:
    """Three-layer guardrail: regex pre-filter -> calibrated model -> review queue."""
    def __init__(self, prob_fn, review_band=(0.4, 0.6)):
        self.prob_fn = prob_fn
        self.low, self.high = review_band

    def predict(self, comment_text: str) -> dict:
        category, _ = regex_filter(comment_text)
        if category is not None:
            return {"decision": "block", "confidence": 1.0,
                    "triggered_layer": 1, "category": category}

        prob = self.prob_fn(comment_text)
        if prob >= self.high:
            decision = "block"
        elif prob <= self.low:
            decision = "pass"
        else:
            decision = "review"
        return {"decision": decision, "confidence": prob,
                "triggered_layer": 3 if decision == "review" else 2}

pipeline = ModerationPipeline(calibrated_prob)

demos = [
    "I will kill you tonight",                      # layer 1 block
    "you should kill yourself loser",               # layer 1 block (self_harm)
    "this is the most disgusting trash i've seen",  # layer 2 block (model)
    "i'm not sure what to think about this",        # layer 3 review
    "thanks for the recipe, looks great",           # layer 2 pass
]
for text in demos:
    print(text, "->", pipeline.predict(text))

In [ ]:
# --- Pipeline analysis on 1,000 randomly selected eval comments ------------
sample_1k = eval_df.sample(n=1000, random_state=SEED).reset_index(drop=True)

# Vectorized layer-2 calibration so we don't tokenize 1k texts one-at-a-time.
sample_raw = distilbert_probs(sample_1k["comment_text"].astype(str).tolist())
sample_cal = isotonic.predict(sample_raw)

results = []
for text, p in zip(sample_1k["comment_text"], sample_cal):
    cat, _ = regex_filter(str(text))
    if cat is not None:
        results.append({"decision": "block", "confidence": 1.0,
                        "triggered_layer": 1, "category": cat})
    elif p >= 0.6:
        results.append({"decision": "block", "confidence": p, "triggered_layer": 2})
    elif p <= 0.4:
        results.append({"decision": "pass", "confidence": p, "triggered_layer": 2})
    else:
        results.append({"decision": "review", "confidence": p, "triggered_layer": 3})

results_df = pd.DataFrame(results)
results_df["label"] = sample_1k["label"].values

# Layer-distribution summary.
layer_counts = results_df["triggered_layer"].value_counts().sort_index()
decision_counts = results_df["decision"].value_counts()
regex_breakdown = results_df.loc[results_df["triggered_layer"] == 1, "category"].value_counts()
print("layer distribution (n=1000):"); print(layer_counts)
print("\ndecision distribution:"); print(decision_counts)
print("\nregex categories triggered:"); print(regex_breakdown)

In [ ]:
# --- Threshold-band sensitivity: try alternative review windows ------------
# Reuses the cached `sample_cal` (calibrated probs on the 1k sample) so we
# don't re-tokenize for every band.
def evaluate_band(low: float, high: float):
    decisions = []
    for text, p in zip(sample_1k["comment_text"], sample_cal):
        cat, _ = regex_filter(str(text))
        if cat is not None:
            decisions.append(("block", 1, p))
        elif p >= high:
            decisions.append(("block", 2, p))
        elif p <= low:
            decisions.append(("pass", 2, p))
        else:
            decisions.append(("review", 3, p))
    df = pd.DataFrame(decisions, columns=["decision", "layer", "prob"])
    df["label"] = sample_1k["label"].values

    auto = df[df["decision"].isin(["block", "pass"])]
    auto_y = auto["label"].values
    auto_pred = (auto["decision"] == "block").astype(int).values
    return {
        "review_n": int((df["decision"] == "review").sum()),
        "auto_n": len(auto),
        "auto_F1": round(f1_score(auto_y, auto_pred, zero_division=0), 4),
        "auto_P": round(precision_score(auto_y, auto_pred, zero_division=0), 4),
        "auto_R": round(recall_score(auto_y, auto_pred, zero_division=0), 4),
    }

bands = [(0.45, 0.55), (0.4, 0.6), (0.3, 0.7)]
sens = pd.DataFrame({f"[{lo},{hi}]": evaluate_band(lo, hi) for lo, hi in bands}).T
print(sens)
# Trade-off: narrower band -> fewer human reviews + cheaper, but auto-action
# F1 drops as more borderline cases get auto-decided. Wider band does the
# inverse: more human cost, higher auto-action precision/recall.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Auto-actioned subset = layers 1 + 2 (excludes the human review queue).
auto = results_df[results_df["decision"].isin(["block", "pass"])]
auto_y = auto["label"].values
auto_pred = (auto["decision"] == "block").astype(int).values
print(f"auto-actioned: n={len(auto)}  "
      f"P={precision_score(auto_y, auto_pred, zero_division=0):.3f}  "
      f"R={recall_score(auto_y, auto_pred, zero_division=0):.3f}  "
      f"F1={f1_score(auto_y, auto_pred, zero_division=0):.3f}")

# Review queue: ground-truth toxic/non-toxic split tells us how much genuinely
# ambiguous content we're catching.
review = results_df[results_df["decision"] == "review"]
print(f"\nreview queue: n={len(review)}  "
      f"toxic={(review['label'] == 1).sum()}  "
      f"non-toxic={(review['label'] == 0).sum()}")

# Layer-distribution plot.
plt.figure(figsize=(6, 4))
sns.barplot(x=layer_counts.index, y=layer_counts.values)
plt.xlabel("triggered layer"); plt.ylabel("count"); plt.title("Pipeline layer distribution (n=1000)")
plt.tight_layout(); plt.show()

### Choosing the human-review band

The sensitivity table above shows the trade-off directly:

- **`[0.45, 0.55]` (narrow)** — minimizes human-review volume. Auto-action F1 falls because borderline cases get a coin-flip auto-decision.
- **`[0.4, 0.6]` (default)** — middle ground. Catches the bulk of low-confidence predictions for review without flooding the queue.
- **`[0.3, 0.7]` (wide)** — maximizes auto-action precision and recall, but the review queue grows rapidly. Operationally expensive at scale.

I keep `[0.4, 0.6]` as the default. For a real platform this should be retuned to operations capacity: pick the widest band the human team can clear within the SLA, which lifts auto-action quality on the rest.